# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/taqadussana/ML/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Unit of analysis:** one row = one content page, for one client, on one day
(the grain of `fact_content_daily_performance`) —confirmed below, zero duplicate rows
found for any client+content+date combination in this slice.

**Time window:** month = 2026-03 (a mid-panel month), deliberately avoiding the `_sample`
table (fixed to June 2026, the natural future outcome window). Verified: 9,841,378 rows,
spanning exactly 2026-03-01 to 2026-03-31.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Features (knowable before the decision point):**
`avg_position`, `impressions`, `days_since_last_update`, `ctr`, `engagement_rate`

**Label/proxy:** `trend_direction == "down"` → `is_declining_label` — the same proxy from
Week 1/2, now built from real daily warehouse data instead of the small starter CSV.

**Context (background, not fed to the model):** `client_hash_id`, `content_hash_id` — used
only for joins and grouping, never as a feature (the ids themselves carry no signal).

**Excluded, with why:** any raw query, URL, or title text — even pseudonymized, these fields
risk exposing real client information if ever displayed publicly, and they're not needed for
a scoring/ranking task built on numeric signals.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
con.sql(f"""
DESCRIBE SELECT * FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet') LIMIT 1
""").df()

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import duckdb, os
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{os.environ['HF_TOKEN']}')")

rel = "hf://datasets/FlyRank/internship-warehouse"

# Grain check: one row per client_hash_id + content_hash_id + report_date?
con.sql(f"""
SELECT client_hash_id, content_hash_id, report_date, COUNT(*) AS n
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01'
GROUP BY 1,2,3
HAVING COUNT(*) > 1
LIMIT 5
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,report_date,n


In [8]:
# Row count and date span for March 2026
con.sql(f"""
SELECT COUNT(*) AS row_count, MIN(report_date) AS min_date, MAX(report_date) AS max_date
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01'
""").df()

,row_count,min_date,max_date
0,9841378,2026-03-01,2026-03-31


In [9]:
# Availability check with IS TRUE
con.sql(f"""
SELECT COUNT(*) AS total_rows,
       COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01'
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,ga4_available_rows
0,9841378,413966


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**Data limits what this slice can never tell me:**

This slice covers exactly one month (2026-03), so it can't tell me about seasonality —
whether a page's movement this month reflects a real trend or just this month's calendar
quirks. It also can't distinguish a genuine decline from consolidation (a sibling page on
the same site simply absorbing the traffic this page lost) without a separate check across
related content groups.

The panel is unbalanced across clients — some have far more tracking history than others —
and GA4 coverage is sparse: verified in this slice, only **413,966 of 9,841,378 rows
(about 4.2%)** have `ga4_data_available = TRUE`. That means any feature relying on GA4
signals (sessions, engagement rate, scroll rate) has very thin coverage this month, and
rows without GA4 data are search-only — not "no traffic," just "not tracked yet."

Because of this, everything in this notebook is observed, single-month evidence — not a
claim about year-round behavior, and not proof that a page's decline is permanent or
causally linked to anything actionable yet.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.